In [1]:
!pip install transformers scikit-learn pandas

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel
import json

In [3]:
# ---------- LOAD DATA ----------
def load_data(path):
    df = pd.read_csv(path, sep="\t", header=None)
    df.columns = ["text", "labels", "id"]
    return df

train_df = load_data("/content/train.tsv")
dev_df   = load_data("/content/dev.tsv")
test_df  = load_data("/content/test.tsv")


# ---------- LOAD LABEL NAMES ----------
with open("/content/emotions.txt") as f:
    emotions = [line.strip() for line in f.readlines()]

label2emotion = {i: e for i, e in enumerate(emotions)}


# ---------- LOAD EKMAN MAPPING ----------
with open("/content/ekman_mapping.json") as f:
    ekman_map = json.load(f)

# 🔥 FIX: reverse mapping
inverse_ekman = {}
for ekman_label, emo_list in ekman_map.items():
    for emo in emo_list:
        inverse_ekman[emo] = ekman_label

In [4]:
# ---------- DEFINE TARGET LABELS ----------
ekman_classes = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]


# ---------- PARSE LABELS (FIXED FOR YOUR DATA) ----------
def parse_labels(x):
    try:
        return [int(x)]   # single-label → list
    except:
        return []

In [5]:
# ---------- MAP TO EKMAN ----------
def map_to_ekman(label_list):
    out = set()

    for l in label_list:
        emo = label2emotion.get(l, None)
        if emo in inverse_ekman:
            out.add(inverse_ekman[emo])

    return list(out)


# ---------- MULTI-LABEL VECTOR ----------
def to_vector(labels):
    vec = np.zeros(len(ekman_classes))
    for l in labels:
        if l in ekman_classes:
            vec[ekman_classes.index(l)] = 1
    return vec

In [6]:
# ---------- PROCESS FUNCTION ----------
def process_df(df):
    df = df.copy()

    df["label_list"] = df["labels"].apply(parse_labels)
    df["ekman"] = df["label_list"].apply(map_to_ekman)
    df["final_labels"] = df["ekman"].apply(to_vector)

    # remove rows with no Ekman mapping
    df = df[df["final_labels"].apply(lambda x: x.sum() > 0)]

    return df

In [7]:
# ---------- APPLY ----------
train_df = process_df(train_df)
dev_df   = process_df(dev_df)
test_df  = process_df(test_df)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
from collections import Counter

def build_vocab(texts, max_vocab=20000):
    counter = Counter()

    for text in texts:
        tokens = text.lower().split()
        counter.update(tokens)

    vocab = {word: i+2 for i, (word, _) in enumerate(counter.most_common(max_vocab))}

    vocab["<PAD>"] = 0
    vocab["<UNK>"] = 1

    return vocab

vocab = build_vocab(train_df["text"])

In [9]:
from torch.utils.data import Dataset

class BERTDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=64):
        self.texts = df["text"].tolist()
        self.labels = df["final_labels"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float32)
        }

In [10]:
class BERTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained("distilbert-base-uncased")
        self.fc = nn.Linear(768, 6)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]
        return self.fc(cls)  # ❌ NO sigmoid

In [11]:
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
labels = np.array(train_df["final_labels"].tolist())

pos_weights = (len(labels) - labels.sum(axis=0)) / (labels.sum(axis=0) + 1e-6)

pos_weights = torch.tensor(pos_weights, dtype=torch.float32).to(device)

print("Class weights:", pos_weights)

Class weights: tensor([ 5.0560, 46.1586, 44.6019,  0.8177, 10.0726,  5.6099], device='cuda:0')


In [12]:
!pip install tqdm

In [13]:
from tqdm import tqdm

def train_epoch(model, loader, optimizer, device, criterion):
    model.train()
    total_loss = 0

    loop = tqdm(loader, desc="Training", leave=False)

    for batch in loop:
        labels = batch["labels"].to(device)

        if "attention_mask" in batch:
            outputs = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device)
            )
        else:
            outputs = model(batch["input_ids"].to(device))

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # update progress bar
        loop.set_postfix(loss=loss.item())

    return total_loss / len(loader)

In [14]:
def evaluate(model, loader, device):
    model.eval()
    preds, true = [], []

    loop = tqdm(loader, desc="Evaluating", leave=False)

    with torch.no_grad():
        for batch in loop:
            labels = batch["labels"].cpu().numpy()

            if "attention_mask" in batch:
                outputs = model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device)
                )
            else:
                outputs = model(batch["input_ids"].to(device))

            outputs = torch.sigmoid(outputs).cpu().numpy()
            pred = (outputs > 0.5).astype(int)

            preds.extend(pred)
            true.extend(labels)

    micro = f1_score(true, preds, average="micro")
    macro = f1_score(true, preds, average="macro")

    return micro, macro

In [16]:
# ===== BERT ONLY =====

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# 1. Dataset + loaders
bert_train_loader = DataLoader(BERTDataset(train_df, tokenizer), batch_size=16, shuffle=True)
bert_dev_loader   = DataLoader(BERTDataset(dev_df, tokenizer), batch_size=16)

# 2. Model
bert_model = BERTModel().to(device)

# 3. Optimizer + loss
optimizer = torch.optim.AdamW(bert_model.parameters(), lr=2e-5)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

# 4. Train
for epoch in range(12):
    loss = train_epoch(bert_model, bert_train_loader, optimizer, device, criterion)
    micro, macro = evaluate(bert_model, bert_dev_loader, device)

    print(f"[BERT] Epoch {epoch+1}")
    print("Loss:", loss, "Micro:", micro, "Macro:", macro)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[BERT] Epoch 1
Loss: 0.5883224760397904 Micro: 0.6725997842502697 Macro: 0.5343150901060655


[BERT] Epoch 2
Loss: 0.38334089312550157 Micro: 0.742676622630672 Macro: 0.6047904331923789


[BERT] Epoch 3
Loss: 0.2805244227673361 Micro: 0.7658485167896402 Macro: 0.6405271152391775


...

[BERT] Epoch 8
Loss: 0.056840044178724625 Micro: 0.7887459807073955 Macro: 0.6638581323040446


[BERT] Epoch 9
Loss: 0.048502964603623286 Micro: 0.7660940760956815 Macro: 0.6339180597505522


[BERT] Epoch 10
Loss: 0.04417792200053755 Micro: 0.7664728682170543 Macro: 0.6381963237899994


[BERT] Epoch 11
Loss: 0.038263380572064055 Micro: 0.797634691195795 Macro: 0.6841614042870935


[BERT] Epoch 12
Loss: 0.030662478543963912 Micro: 0.7980132450331126 Macro: 0.6861957319324677


In [17]:
torch.save({
    "model_state_dict": bert_model.state_dict(),
    "pos_weights": pos_weights,
    "labels": ekman_classes
}, "full_model.pt")

In [18]:
from sklearn.metrics import f1_score, accuracy_score

def evaluate(model, loader, device):
    model.eval()
    preds, true = [], []

    with torch.no_grad():
        for batch in loader:
            labels = batch["labels"].cpu().numpy()

            if "attention_mask" in batch:
                outputs = model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device)
                )
            else:
                outputs = model(batch["input_ids"].to(device))

            outputs = torch.sigmoid(outputs).cpu().numpy()
            pred = (outputs > 0.5).astype(int)

            preds.extend(pred)
            true.extend(labels)

    preds = np.array(preds)
    true = np.array(true)

    # F1
    micro = f1_score(true, preds, average="micro")
    macro = f1_score(true, preds, average="macro")

    # Accuracy (subset)
    subset_acc = accuracy_score(true, preds)

    # Accuracy (element-wise)
    element_acc = (preds == true).mean()

    return micro, macro, subset_acc, element_acc

In [22]:
micro, macro, subset_acc, element_acc = evaluate(bert_model, bert_dev_loader, device)

print(f"[BERT] Epoch {epoch+1}")
print("Loss:", loss)
print("Micro F1:", micro)
print("Macro F1:", macro)
print("Subset Acc:", subset_acc)
print("Element Acc:", element_acc)

[BERT] Epoch 1
Loss: 0.030662478543963912
Micro F1: 0.794146662282144
Macro F1: 0.6730975129743343
Subset Acc: 0.7689445196211097
Element Acc: 0.9294091114118178


In [23]:
bert_test_loader = DataLoader(BERTDataset(test_df, tokenizer), batch_size=16)

In [24]:
micro, macro, subset_acc, element_acc = evaluate(bert_model, bert_test_loader, device)

df = pd.DataFrame([{
    "model": "BERT",
    "micro_f1": micro,
    "macro_f1": macro,
    "subset_accuracy": subset_acc,
    "element_accuracy": element_acc
}])

df.to_csv("bert_final_results.csv", index=False)

In [26]:
import pandas as pd

df = pd.DataFrame([{
    "epoch": epoch + 1,
    "loss": loss,
    "micro_f1": micro,
    "macro_f1": macro,
    "subset_accuracy": subset_acc,
    "element_accuracy": element_acc
}])

df.to_csv("bert_trainset_result.csv", index=False)

print("Saved current results.")

Saved current results.
